In [ ]:
!pip install sahi
!pip install -U ultralytics
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.12.0.88
    Uninstalling opencv-python-4.12.0.88:
      Successfully uninstalled opencv-python-4.12.0.88
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.1 MB/s eta 0:00:00


In [ ]:
import os
import gc
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.windows import Window
from shapely.geometry import Point
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO
from scipy.spatial import cKDTree
from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


eksport modelu (musi być w tej samej sesji)

In [ ]:
model = YOLO("/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt")

model.export(
    format='engine',
    imgsz=1024,
    half=True,
    device=0,
    workspace=4,
    simplify=True
)

Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n-obb summary (fused): 109 layers, 2,653,918 parameters, 0 gradients, 6.6 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 6, 21504) (5.7 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 14 packages in 215ms
Prepared 6 packages in 4.97s
Installed 6 packages in 372ms
 + colorama==0.4.6
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnx==1.20.0
 + onnxruntime-gpu==1.23.2
 + onnxslim==0.1.82

requirements: AutoUpdate success ✅ 6.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to 

'/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine'

Główna funkcja

In [ ]:
def process_geoportal_year_chunked(folder_name, model_path, output_filename="detected_vehicles", confidence_threshold=0.25):
    """
    Przetwarza pliki .tif kawałkami (chunks), aby uniknąć przepełnienia RAM.
    """
    base_dir = f"/content/drive/MyDrive/dane_detekcja_pojazdow/dane_geoportal/{folder_name}"

    if not os.path.exists(base_dir):
        print(f"Błąd: Ścieżka {base_dir} nie istnieje!")
        return

    # Inicjalizacja modelu 
    print(f"Loading TensorRT model: {model_path}...")
    yolo_model = YOLO(model_path, task="obb")

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model=yolo_model,
        device="cuda:0",
        confidence_threshold=confidence_threshold,
        image_size=1024
    )

    all_detections = []
    tif_files = [f for f in os.listdir(base_dir) if f.lower().endswith('.tif')]
    print(f"Found {len(tif_files)} files in folder: {folder_name}")

    # Ustawienia kafelkowania ręcznego (czytanie z dysku)
    CHUNK_SIZE = 20480  # Rozmiar kawałka wczytywanego do RAM (np. 4096x4096)
    OVERLAP = 128      # Zakładka między kawałkami, aby nie uciąć aut na krawędzi

    for file_name in tif_files:
        image_path = os.path.join(base_dir, file_name)
        print(f"\n--- Processing file: {file_name} ---")

        with rasterio.open(image_path) as src:
            width = src.width
            height = src.height
            transform = src.transform
            crs = src.crs
            print(f"Dimensions: {width}x{height}, CRS: {crs}")

            # Pętla po oknie (kafelkach)
            for y in range(0, height, CHUNK_SIZE - OVERLAP):
                for x in range(0, width, CHUNK_SIZE - OVERLAP):
                    # Definicja okna do odczytu
                    window = Window(x, y, CHUNK_SIZE, CHUNK_SIZE)

                    # Odczyt tylko fragmentu obrazu do RAM
                    # Rasterio zwraca (C, H, W), SAHI potrzebuje (H, W, C)
                    img_chunk = src.read(window=window)

                    # Jeśli okno wyszło poza obraz, rasterio zwróci mniejszy fragment
                    actual_h, actual_w = img_chunk.shape[1], img_chunk.shape[2]
                    if actual_h == 0 or actual_w == 0:
                        continue

                    # Konwersja formatu: (Channels, Height, Width) -> (Height, Width, Channels)
                    img_chunk = np.moveaxis(img_chunk, 0, -1)

                    # Inferencja na małym wycinku
                    # Używamy get_sliced_prediction nawet na wycinku, żeby SAHI obsłużyło detekcję "wewnątrz" wycinka
                    result = get_sliced_prediction(
                        img_chunk,
                        detection_model,
                        slice_height=1024,
                        slice_width=1024,
                        overlap_height_ratio=0.2,
                        overlap_width_ratio=0.2,
                        perform_standard_pred=False,
                        verbose=0
                    )

                    # Przeliczanie współrzędnych
                    for prediction in result.object_prediction_list:
                        # Współrzędne względem wycinka (chunk)
                        local_x = (prediction.bbox.minx + prediction.bbox.maxx) / 2
                        local_y = (prediction.bbox.miny + prediction.bbox.maxy) / 2

                        # Współrzędne względem całego obrazu (piksele)
                        global_pixel_x = x + local_x
                        global_pixel_y = y + local_y

                        # Transformacja do układu współrzędnych mapy (metry)
                        world_x, world_y = transform * (global_pixel_x, global_pixel_y)

                        all_detections.append({
                            'geometry': Point(world_x, world_y),
                            'confidence': prediction.score.value,
                            'class_name': prediction.category.name,
                            'image_source': file_name,
                            'year_folder': folder_name
                        })

                    # Zwolnienie pamięci po każdym kawałku
                    del img_chunk
                    del result

            # Garbage collector co plik (lub częściej jeśli trzeba)
            gc.collect()

    # Zapis wyniku
    if all_detections:
        print(f"Initial detection count: {len(all_detections)}. Starting deduplication...")

        # 1. Tworzymy GeoDataFrame
        gdf = gpd.GeoDataFrame(all_detections, crs='EPSG:2180')

        # 2. Sortujemy po confidence malejąco (najważniejszy krok!)
        # Dzięki temu pierwszy punkt w parze zawsze będzie tym "lepszym"
        gdf = gdf.sort_values(
            by='confidence',
            ascending=False
        ).reset_index(drop=True)

        # 3. Przygotowanie danych do szybkiego wyszukiwania (cKDTree)
        # Wyciągamy współrzędne do macierzy numpy (dużo szybsze niż operacje na geometrii)
        coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
        tree = cKDTree(coords)

        # 4. Znajdź wszystkie pary punktów w odległości <= 1.8m
        # query_pairs zwraca zbiór krotek (i, j) gdzie i < j.
        # Ponieważ posortowaliśmy df po confidence malejąco, 'i' ma zawsze większe (lub równe) confidence niż 'j'.
        duplicates = tree.query_pairs(r=1.8)

        indices_to_drop = set()
        indices_with_overlap = set()

        for i, j in duplicates:
            # i = punkt o wyższym confidence (zostaje)
            # j = punkt o niższym confidence (do usunięcia)
            indices_to_drop.add(j)
            indices_with_overlap.add(i)

        # 5. Dodanie flagi informacyjnej dla punktów, które "wygrały"
        # Domyślnie False, ustawiamy True tylko dla tych, które miały konkurenta
        gdf['overlap_resolved'] = False

        # Uważamy, żeby nie oznaczyć punktu, który sam zaraz zostanie usunięty (bo przegrał z kimś jeszcze lepszym)
        # Zbiór wygranych to ci, którzy mieli konflikt, MINUS ci, którzy sami są do usunięcia
        real_winners = indices_with_overlap - indices_to_drop
        gdf.loc[list(real_winners), 'overlap_resolved'] = True

        # 6. Usuwanie duplikatów
        gdf_clean = gdf.drop(index=list(indices_to_drop))

        print(f"Removed {len(indices_to_drop)} duplicates. Final count: {len(gdf_clean)}.")

        # Zapis
        output_dir = "/content/drive/MyDrive/dane_detekcja_pojazdow/warstwy_wektorowe"
        output_path = f"{output_dir}/{output_filename}_{folder_name}.geojson"

        if not os.path.exists(output_dir):
            print(f"Creating missing directory: {output_dir}")
            os.makedirs(output_dir, exist_ok=True)

        gdf_clean.to_file(output_path, driver='GeoJSON')
        print(f"✅ SUCCESS! Vector layer saved to: {output_path}")
    else:
        print("No objects detected.")

# Wykorzystanie modelu - utworzenie warstw

In [ ]:
ENGINE_MODEL_PATH = "/content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine"

# Wywołanie dla konkretnego folderu
process_geoportal_year_chunked(
    folder_name="rok_2025",
    model_path=ENGINE_MODEL_PATH,
    output_filename="vehicle_points",
    confidence_threshold=0.136
)

Loading TensorRT model: /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Found 4 files in folder: rok_2025

--- Processing file: 83139_1483340_M-34-64-D-d-1-4.tif ---
Dimensions: 45301x46866, CRS: EPSG:2180
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...

--- Processing file: 83139_1483338_M-34-64-D-d-1-2.tif ---
Dimensions: 45282x46866, CRS: PROJCS["ETRF2000-PL / CS92",GEOGCS["ETRF2000-PL",DATUM["ETRF2000_Poland",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1305"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORI

In [ ]:
years = ["rok_2013", "rok_2021", "rok_2023_marzec", "rok_2023_wrzesien"]
for year in years:
    if year == "rok_2013":
      treshold = 0.163
    else:
      treshold = 0.136

    process_geoportal_year_chunked(
        folder_name=year,
        model_path=ENGINE_MODEL_PATH,
        output_filename="vehicle_points",
        confidence_threshold=treshold
    )

Loading TensorRT model: /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...
Found 4 files in folder: rok_2013

--- Processing file: 241_181004_M-34-64-D-d-2-3.tif ---


Dimensions: 22661x23442, CRS: PROJCS["Transverse Mercator; WGS84; WGS84",GEOGCS["World Geodetic System 1984",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["unknown",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",19],PARAMETER["scale_factor",0.9993],PARAMETER["false_easting",500000],PARAMETER["false_northing",-5300000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Loading /content/drive/MyDrive/YOLO_Project/test_scale_1_0/weights/best.engine for TensorRT inference...

--- Processing file: 241_181001_M-34-64-D-d-1-4.tif ---


Dimensions: 22651x23433, CRS: PROJCS["Transverse Mercator; WGS84; WGS84",GEOGCS["World Geodetic System 1984",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["unknown",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",19],PARAMETER["scale_factor",0.9993],PARAMETER["false_easting",500000],PARAMETER["false_northing",-5300000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

--- Processing file: 241_181002_M-34-64-D-d-2-1.tif ---
Dimensions: 22651x23442, CRS: PROJCS["Transverse Mercator; WGS84; WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PAR